# Goal To Reach

In [ ]:
from transformers import pipeline

In [ ]:
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


In [ ]:
text = "Purchased 5 Pencils worth $50 CAD."
labels = ["Stationary", "Electronics", "Groceries", "Travel", "Food", "Clothing", "Healthcare", "Utilities"]
result = classifier(text, labels)
print(result)

{'sequence': 'Purchased 5 Pencils worth $50 CAD.', 'labels': ['Stationary', 'Utilities', 'Travel', 'Electronics', 'Healthcare', 'Clothing', 'Food', 'Groceries'], 'scores': [0.5581831932067871, 0.11163900047540665, 0.08028705418109894, 0.0760575607419014, 0.061547908931970596, 0.040977995842695236, 0.038523975759744644, 0.032783232629299164]}


# Anatole Farber - Implementation

#### Write your observation: [Classification Tracker](https://docs.google.com/spreadsheets/d/1Pb3ryHwrBZLUZXm_x_-a0evyA-EOY4acN7AHCLOHrwE/edit?usp=sharing)

# Andrew Chen - Implementation

#### Write your observation: [Classification Tracker](https://docs.google.com/spreadsheets/d/1Pb3ryHwrBZLUZXm_x_-a0evyA-EOY4acN7AHCLOHrwE/edit?usp=sharing)

# Kenny Hoang - Implementation

#### Write your observation: [Classification Tracker](https://docs.google.com/spreadsheets/d/1Pb3ryHwrBZLUZXm_x_-a0evyA-EOY4acN7AHCLOHrwE/edit?usp=sharing)

# Lily S. Nickoshie - Implementation

#### Write your observation: [Classification Tracker](https://docs.google.com/spreadsheets/d/1Pb3ryHwrBZLUZXm_x_-a0evyA-EOY4acN7AHCLOHrwE/edit?usp=sharing)

# Nishit Rathod - Implementation

In [1]:
import os
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
import joblib

In [2]:
# 1. Load your synthetic dataset
df = pd.read_csv("https://raw.githubusercontent.com/OneNishitRathod/Text-Description-Classification/refs/heads/Dev/synthetic_transactions_diverse.csv")

In [3]:
df.head()

,text_description,category
0,Procured legal fee and contract review for €36...,Legal
1,"Bought annual fee, membership for CAD$1,244.45...",Subscriptions
2,"Spent onn charger, laptop, mouse for $602.94, ...",Electronics
3,Bought workshop and course fee for ₹286 and JP...,Training
4,"Purchased diesel, petrol, gasoline (incl. gaso...",Fuel


In [4]:
# 2. Encode labels
le = LabelEncoder()
df["label"] = le.fit_transform(df["category"])

In [5]:
# Create models folder if it doesn't exist
os.makedirs("models", exist_ok=True)

joblib.dump(le, "models/label_encoder.joblib")

['models/label_encoder.joblib']

In [6]:
# 3. Convert to HuggingFace Dataset
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2)
train_dataset = dataset["train"]
test_dataset = dataset["test"]

In [7]:
# 4. Tokenize text
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_fn(batch):
    return tokenizer(batch["text_description"], padding="max_length", truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize_fn, batched=True)
test_dataset = test_dataset.map(tokenize_fn, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/12000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [8]:
# 5. Load model
num_labels = len(le.classes_)
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=num_labels)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
# 6. Define metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")
    return {"accuracy": acc, "f1": f1}

In [10]:
# 7. Training arguments
training_args = TrainingArguments(
    output_dir="./models/distilbert_txn",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
)

In [14]:
# 8. Train
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

/tmp/ipython-input-1757797722.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


<IPython.core.display.Javascript object>

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: onenishitrathod (onenishitrathod-humber-college) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.013500,0.012595,0.998000,0.998002
2,0.009300,0.009090,0.998000,0.998001
3,0.001400,0.010122,0.997667,0.997669


TrainOutput(global_step=4500, training_loss=0.10379222309589387, metrics={'train_runtime': 507.4257, 'train_samples_per_second': 70.946, 'train_steps_per_second': 8.868, 'total_flos': 1192610552832000.0, 'train_loss': 0.10379222309589387, 'epoch': 3.0})

In [15]:
# 9. Save model
trainer.save_model("./models/distilbert_txn")
tokenizer.save_pretrained("./models/distilbert_txn")

('./models/distilbert_txn/tokenizer_config.json',
 './models/distilbert_txn/special_tokens_map.json',
 './models/distilbert_txn/vocab.txt',
 './models/distilbert_txn/added_tokens.json',
 './models/distilbert_txn/tokenizer.json')

#### Prediction

In [16]:
import torch, numpy as np, joblib

tokenizer = AutoTokenizer.from_pretrained("models/distilbert_txn")
model = AutoModelForSequenceClassification.from_pretrained("models/distilbert_txn")
le = joblib.load("models/label_encoder.joblib")

def predict_topk_transformer(text, k=3):
    inputs = tokenizer(text, truncation=True, padding=True, return_tensors="pt", max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[0].cpu().numpy()
        probs = np.exp(logits) / np.sum(np.exp(logits))
    topk = np.argsort(probs)[::-1][:k]
    return list(zip(le.inverse_transform(topk), probs[topk]))

print(predict_topk_transformer("Purchased car for taxi company.", k=3))

[('Travel', np.float32(0.9987763)), ('Electronics', np.float32(0.0001831438)), ('Fuel', np.float32(0.00015954966))]


#### Write your observation: [Classification Tracker](https://docs.google.com/spreadsheets/d/1Pb3ryHwrBZLUZXm_x_-a0evyA-EOY4acN7AHCLOHrwE/edit?usp=sharing)

# Yanzhen Zhang - Implementation

#### Write your observation: [Classification Tracker](https://docs.google.com/spreadsheets/d/1Pb3ryHwrBZLUZXm_x_-a0evyA-EOY4acN7AHCLOHrwE/edit?usp=sharing)